# DriveSense-VLM — 04: Full 4-Level Evaluation

Runs the complete evaluation framework:
- **Level 1** — Grounding accuracy (IoU + Hungarian matching)
- **Level 2** — Reasoning quality (LLM-as-judge via Claude API)
- **Level 3** — Production benchmarks (latency, VRAM, quantization degradation)
- **Level 4** — Robustness (stratified by time-of-day, weather, location, OOD)

**GPU required**: A100 for prediction generation; T4 sufficient for Level 2 judging  
**Estimated time**: ~1 hour  
**Estimated cost**: ~12 CU  

**Prerequisites**: `01_training.ipynb` (checkpoint), `03_benchmark.ipynb` (benchmark JSON for Level 3).

## ⚠️ Before Running

- Set **Runtime → Change runtime type → A100 GPU** for prediction generation (Cell 4).
- For Level 2, you need an `ANTHROPIC_API_KEY` **or** use `--mock-judge` to skip real judging.
- Level 3 reads the JSON files from `03_benchmark.ipynb` — no GPU needed for that level.

In [ ]:
# Cell 3: Mount Drive + clone repo + symlinks + install
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

!pip install -e '.[eval]' -q
print('Setup complete.')

In [ ]:
# Cell 4: Generate predictions on the test set
# Requires A100 GPU + trained checkpoint
import glob
from pathlib import Path

training_dir = Path(PROJECT_ROOT) / 'outputs' / 'training'
best = training_dir / 'checkpoint-best'
if not best.exists():
    ckpts = sorted(glob.glob(str(training_dir / 'checkpoint-*')))
    best = Path(ckpts[-1]) if ckpts else None

if best is None:
    print('No checkpoint found — using --mock for prediction generation.')
    MOCK_FLAG = '--mock'
else:
    print(f'Using checkpoint: {best}')
    MOCK_FLAG = ''

!python scripts/run_generate_predictions.py \
    --split test \
    --adapter-path {best if best else ''} \
    --output {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    {MOCK_FLAG}

pred_path = Path(PROJECT_ROOT) / 'outputs' / 'predictions' / 'test_predictions.jsonl'
if pred_path.exists():
    with open(pred_path) as f:
        n = sum(1 for _ in f)
    print(f'Predictions generated: {n} examples → {pred_path}')

In [ ]:
# Cell 5: Level 1 — Grounding accuracy (IoU + Hungarian matching)
# No API key required.
!python scripts/run_evaluation.py \
    --level 1 \
    --predictions {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    --output-dir {PROJECT_ROOT}/outputs/eval/level1

import json
from pathlib import Path

l1 = Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'level1' / 'grounding_metrics.json'
if l1.exists():
    m = json.loads(l1.read_text())
    print('\nLevel 1 — Grounding Metrics:')
    for k, v in m.items():
        print(f'  {k}: {v}')

In [ ]:
# Cell 6: Level 2 — Reasoning quality (LLM-as-judge)
# Set ANTHROPIC_API_KEY for real judging, or keep --mock-judge for free run.
import os

# Uncomment and set your key for real LLM judging:
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

USE_MOCK_JUDGE = 'ANTHROPIC_API_KEY' not in os.environ
mock_judge_flag = '--mock-judge' if USE_MOCK_JUDGE else ''
if USE_MOCK_JUDGE:
    print('ANTHROPIC_API_KEY not set — using mock judge (random scores in [3,5]).')

!python scripts/run_evaluation.py \
    --level 2 \
    --predictions {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    --output-dir {PROJECT_ROOT}/outputs/eval/level2 \
    {mock_judge_flag}

l2 = Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'level2' / 'reasoning_metrics.json'
if l2.exists():
    m = json.loads(l2.read_text())
    print('\nLevel 2 — Reasoning Metrics:')
    for k, v in m.items():
        print(f'  {k}: {v}')

In [ ]:
# Cell 7: Level 3 — Production metrics (reads benchmark JSON from 03_benchmark.ipynb)
# No GPU needed — reads stored benchmark results.
!python scripts/run_evaluation.py \
    --level 3 \
    --benchmarks-dir {PROJECT_ROOT}/outputs/benchmarks \
    --output-dir {PROJECT_ROOT}/outputs/eval/level3

l3 = Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'level3' / 'production_metrics.json'
if l3.exists():
    m = json.loads(l3.read_text())
    print('\nLevel 3 — Production Metrics:')
    for k, v in m.items():
        print(f'  {k}: {v}')

In [ ]:
# Cell 8: Level 4 — Robustness (stratified by time-of-day, weather, location, OOD)
!python scripts/run_evaluation.py \
    --level 4 \
    --predictions {PROJECT_ROOT}/outputs/predictions/test_predictions.jsonl \
    --output-dir {PROJECT_ROOT}/outputs/eval/level4

l4 = Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'level4' / 'robustness_metrics.json'
if l4.exists():
    m = json.loads(l4.read_text())
    print('\nLevel 4 — Robustness Metrics:')
    for k, v in m.items():
        print(f'  {k}: {v}')

In [ ]:
# Cell 9: Generate final evaluation report
!python scripts/run_full_evaluation.py \
    --generate-report \
    --output-dir {PROJECT_ROOT}/outputs/eval

In [ ]:
# Cell 10: Display final evaluation summary
from pathlib import Path

report_candidates = [
    Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'final_report.txt',
    Path(PROJECT_ROOT) / 'outputs' / 'eval' / 'evaluation_report.txt',
]

report = None
for candidate in report_candidates:
    if candidate.exists():
        report = candidate
        break

if report:
    print(report.read_text())
else:
    # Fall back to printing individual metric files
    eval_dir = Path(PROJECT_ROOT) / 'outputs' / 'eval'
    for p in sorted(eval_dir.rglob('*.json')):
        print(f'\n--- {p.relative_to(eval_dir)} ---')
        print(p.read_text())

In [ ]:
# Cell 11: Verify all eval outputs are on Drive
from pathlib import Path

eval_dir = Path(PROJECT_ROOT) / 'outputs' / 'eval'
print('Evaluation outputs on Drive:')
for p in sorted(eval_dir.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.relative_to(eval_dir)}  ({size_kb:.1f} KB)')

print('\nEvaluation complete! All results saved to Google Drive.')